# 17 · Self-supervised masked-feature pretraining (DuETT's key ingredient, tabular form)

Rounds 1-5: every neural/graph variant lands at 0.905-0.906 test AUROC
(base+temporal, core+aux training), tuned XGBoost at 0.908; PLR embeddings
(notebook 16) did not help. DuETT's ablations (RESEARCH.md prompt 1) credit
most of its +0.026 over XGBoost to masked value + presence pretraining.

Here the encoder is first pretrained to reconstruct randomly masked columns
(values and missingness indicators, 30% mask, MSE) from the FEATURES of the
training rows (core-train + auxiliary patients) -- never labels, never
model_val/test rows -- then fine-tuned with the same 5-fold cross-fitting.
One pretraining per seed; every fold starts from it.

In [ ]:
import os, sys, json, time, glob, shutil, subprocess
import numpy as np, pandas as pd, torch
from sklearn.metrics import average_precision_score

WORK = "/kaggle/working"
for d in ("src", "tests", "processed", "results_v2"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
open(f"{WORK}/src/__init__.py", "w").close()
sys.path.insert(0, WORK)
OUT = f"{WORK}/results_v2"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [42, 43, 44]
print(f"[v2] device={DEV} torch={torch.__version__}", flush=True)

In [ ]:
%%writefile /kaggle/working/src/mimic_concepts.py
"""Clinical concept dictionaries for MIMIC-IV v3.1.

Everything that maps a raw MIMIC identifier to a clinical concept lives here so
`mimic_iv.py` stays pure ETL mechanics. Three groups:

  * `CHART_ITEMS` / `LAB_ITEMS`  -- itemid -> concept, for the streaming
    aggregators over `icu/chartevents.csv` and `hosp/labevents.csv`.
  * `INPUT_ITEMS` / `PROC_ITEMS` / `OUTPUT_ITEMS` -- ICU treatment exposure,
    the "interventions and medications" block of the SBSCGM feature encoder
    (arXiv:2508.00615 sec. 3.2).
  * `CHARLSON_ICD9` / `CHARLSON_ICD10` -- Quan et al. (2005) code-prefix maps
    for the comorbidity block.

Physiologic plausibility ranges are attached to every numeric concept.
MIMIC chart/lab values contain transcription errors (a heart rate of 8000, a
creatinine of -1); clipping to a clinical range before aggregation is standard
practice and stops one bad row from destroying a patient's min/max features.
"""

# --------------------------------------------------------------- ICU vitals
# itemid -> (concept, unit_hint, lo, hi). Both invasive and non-invasive
# pressures map to the same concept: which line a patient has is a treatment
# decision, not a different physiologic quantity.
CHART_ITEMS = {
    220045: ("heart_rate", "bpm", 10, 300),
    220050: ("sbp", "mmHg", 20, 300),
    220179: ("sbp", "mmHg", 20, 300),
    225309: ("sbp", "mmHg", 20, 300),
    220051: ("dbp", "mmHg", 5, 200),
    220180: ("dbp", "mmHg", 5, 200),
    225310: ("dbp", "mmHg", 5, 200),
    220052: ("map", "mmHg", 10, 250),
    220181: ("map", "mmHg", 10, 250),
    225312: ("map", "mmHg", 10, 250),
    220210: ("resp_rate", "bpm", 2, 80),
    224690: ("resp_rate", "bpm", 2, 80),
    220277: ("spo2", "%", 20, 100),
    223761: ("temp_f", "F", 70, 115),
    223762: ("temp_c", "C", 20, 46),
    220739: ("gcs_eye", "pts", 1, 4),
    223900: ("gcs_verbal", "pts", 1, 5),
    223901: ("gcs_motor", "pts", 1, 6),
    220621: ("glucose_chart", "mg/dL", 10, 1500),
    225664: ("glucose_chart", "mg/dL", 10, 1500),
    226512: ("weight_kg", "kg", 20, 400),
    223835: ("fio2", "%", 20, 100),
}

# ------------------------------------------------------------------- labs
LAB_ITEMS = {
    50912: ("creatinine", "mg/dL", 0.1, 25),
    51006: ("bun", "mg/dL", 1, 250),
    50971: ("potassium", "mEq/L", 1, 12),
    50983: ("sodium", "mEq/L", 90, 200),
    50902: ("chloride", "mEq/L", 60, 160),
    50882: ("bicarbonate", "mEq/L", 3, 60),
    50931: ("glucose_lab", "mg/dL", 10, 1500),
    51221: ("hematocrit", "%", 5, 70),
    51222: ("hemoglobin", "g/dL", 2, 25),
    51301: ("wbc", "K/uL", 0.1, 200),
    51265: ("platelets", "K/uL", 5, 1500),
    50813: ("lactate", "mmol/L", 0.2, 30),
    50820: ("ph", "", 6.5, 8.0),
    50821: ("po2", "mmHg", 10, 700),
    50818: ("pco2", "mmHg", 5, 150),
    50802: ("base_excess", "mEq/L", -30, 30),
    51237: ("inr", "", 0.5, 15),
    51275: ("ptt", "s", 10, 200),
    50885: ("bilirubin", "mg/dL", 0.1, 50),
    50862: ("albumin", "g/dL", 0.5, 7),
    50863: ("alk_phos", "IU/L", 10, 2000),
    50861: ("alt", "IU/L", 5, 5000),
    50878: ("ast", "IU/L", 5, 5000),
    50960: ("magnesium", "mg/dL", 0.5, 8),
    50893: ("calcium", "mg/dL", 3, 20),
    50970: ("phosphate", "mg/dL", 0.5, 20),
    51003: ("troponin_t", "ng/mL", 0.0, 30),
    50889: ("crp", "mg/L", 0.0, 400),
}

# --------------------------------------------------- ICU treatment exposure
# Binary "did this patient receive X in the window" flags. These are the
# strongest single predictors in an ICU cohort and are exactly the
# "high-risk intervention" indicators the SBSCGM encoder calls for.
INPUT_ITEMS = {
    221906: "norepinephrine",
    221289: "epinephrine",
    221749: "phenylephrine",
    222315: "vasopressin",
    221662: "dopamine",
    221653: "dobutamine",
    221468: "milrinone",
    222168: "propofol",
    221668: "midazolam",
    221744: "fentanyl",
    225158: "crystalloid_ns",
    225828: "crystalloid_lr",
    220862: "albumin_infusion",
    225168: "prbc_transfusion",
}

# Vasopressors get their own aggregate flag: any of these means shock.
VASOPRESSORS = ("norepinephrine", "epinephrine", "phenylephrine",
                "vasopressin", "dopamine")

PROC_ITEMS = {
    225792: "invasive_ventilation",
    225794: "noninvasive_ventilation",
    227194: "extubation",
    224385: "intubation",
    225802: "crrt",
    225803: "crrt",
    225809: "crrt",
    225955: "crrt",
    225441: "hemodialysis",
    225805: "peritoneal_dialysis",
    224263: "central_line",
    224264: "central_line",
    224267: "arterial_line",
    225752: "arterial_line",
    225454: "chest_tube",
}

# Urine output is charted under a long tail of itemids (Foley, void, ileoconduit
# ...). All of them are summed into a single hourly rate.
# Urine output. 227489 is "GU Irrigant/Urine Volume Out", which includes the
# irrigant that was instilled; 227488 is that instilled volume ("GU Irrigant
# Volume In") and is therefore *subtracted*, matching MIMIC's own
# `urine_output` concept. Summing 227488 as if it were urine inflates output
# for anyone on bladder irrigation.
OUTPUT_URINE_ITEMS = (
    226559, 226560, 226561, 226584, 226563, 226564, 226565,
    226567, 226557, 226558, 227489,
)
OUTPUT_URINE_NEGATIVE_ITEMS = (227488,)

# A single charted void or bag reading above this is a charting error, not a
# measurement. The derived rate is bounded separately in `mimic_iv`.
URINE_EVENT_MAX_ML = 2500.0
# Sustained output above this over a whole observation window is not
# physiologic (brisk post-obstructive diuresis peaks well below it).
URINE_RATE_MAX_ML_PER_HR = 1000.0

# ------------------------------------------------------------- comorbidities
# Quan et al. (2005) ICD code-prefix mapping for the Charlson index. Matched on
# `icd_code.startswith(prefix)` after stripping dots, which is how MIMIC-IV
# stores codes.
CHARLSON_ICD9 = {
    "myocardial_infarction": ("410", "412"),
    "congestive_heart_failure": ("39891", "40201", "40211", "40291", "40401",
                                 "40403", "40411", "40413", "40491", "40493",
                                 "4254", "4255", "4257", "4258", "4259", "428"),
    "peripheral_vascular": ("0930", "4373", "440", "441", "4431", "4432",
                            "4438", "4439", "4471", "5571", "5579", "V434"),
    "cerebrovascular": ("36234", "430", "431", "432", "433", "434", "435",
                        "436", "437", "438"),
    "dementia": ("290", "2941", "3312"),
    "chronic_pulmonary": ("4168", "4169", "490", "491", "492", "493", "494",
                          "495", "496", "500", "501", "502", "503", "504",
                          "505", "5064", "5081", "5088"),
    "rheumatic": ("4465", "7100", "7101", "7102", "7103", "7104", "7140",
                  "7141", "7142", "7148", "725"),
    "peptic_ulcer": ("531", "532", "533", "534"),
    "mild_liver_disease": ("07022", "07023", "07032", "07033", "07044",
                           "07054", "0706", "0709", "570", "571", "5733",
                           "5734", "5738", "5739", "V427"),
    "diabetes_uncomplicated": ("2500", "2501", "2502", "2503", "2508", "2509"),
    "diabetes_complicated": ("2504", "2505", "2506", "2507"),
    "hemiplegia": ("3341", "342", "343", "3440", "3441", "3442", "3443",
                   "3444", "3445", "3446", "3449"),
    "renal_disease": ("40301", "40311", "40391", "40402", "40403", "40412",
                      "40413", "40492", "40493", "582", "5830", "5831",
                      "5832", "5834", "5836", "5837", "585", "586", "5880",
                      "V420", "V451", "V56"),
    "malignancy": ("140", "141", "142", "143", "144", "145", "146", "147",
                   "148", "149", "150", "151", "152", "153", "154", "155",
                   "156", "157", "158", "159", "160", "161", "162", "163",
                   "164", "165", "17", "18", "190", "191", "192", "193",
                   "194", "195", "200", "201", "202", "203", "204", "205",
                   "206", "207", "208", "2386"),
    "severe_liver_disease": ("4560", "4561", "4562", "5722", "5723", "5724",
                             "5728"),
    "metastatic_cancer": ("196", "197", "198", "199"),
    "hiv": ("042", "043", "044"),
}

CHARLSON_ICD10 = {
    "myocardial_infarction": ("I21", "I22", "I252"),
    "congestive_heart_failure": ("I099", "I110", "I130", "I132", "I255",
                                 "I420", "I425", "I426", "I427", "I428",
                                 "I429", "I43", "I50", "P290"),
    "peripheral_vascular": ("I70", "I71", "I731", "I738", "I739", "I771",
                            "I790", "I792", "K551", "K558", "K559", "Z958",
                            "Z959"),
    "cerebrovascular": ("G45", "G46", "H340", "I60", "I61", "I62", "I63",
                        "I64", "I65", "I66", "I67", "I68", "I69"),
    "dementia": ("F00", "F01", "F02", "F03", "F051", "G30", "G311"),
    "chronic_pulmonary": ("I278", "I279", "J40", "J41", "J42", "J43", "J44",
                          "J45", "J46", "J47", "J60", "J61", "J62", "J63",
                          "J64", "J65", "J66", "J67", "J684", "J701", "J703"),
    "rheumatic": ("M05", "M06", "M315", "M32", "M33", "M34", "M351", "M353",
                  "M360"),
    "peptic_ulcer": ("K25", "K26", "K27", "K28"),
    "mild_liver_disease": ("B18", "K700", "K701", "K702", "K703", "K709",
                           "K713", "K714", "K715", "K717", "K73", "K74",
                           "K760", "K762", "K763", "K764", "K768", "K769",
                           "Z944"),
    "diabetes_uncomplicated": ("E100", "E101", "E106", "E108", "E109",
                               "E110", "E111", "E116", "E118", "E119",
                               "E120", "E121", "E126", "E128", "E129",
                               "E130", "E131", "E136", "E138", "E139",
                               "E140", "E141", "E146", "E148", "E149"),
    "diabetes_complicated": ("E102", "E103", "E104", "E105", "E107", "E112",
                             "E113", "E114", "E115", "E117", "E122", "E123",
                             "E124", "E125", "E127", "E132", "E133", "E134",
                             "E135", "E137", "E142", "E143", "E144", "E145",
                             "E147"),
    "hemiplegia": ("G041", "G114", "G801", "G802", "G81", "G82", "G830",
                   "G831", "G832", "G833", "G834", "G839"),
    "renal_disease": ("I120", "I131", "N032", "N033", "N034", "N035", "N036",
                      "N037", "N052", "N053", "N054", "N055", "N056", "N057",
                      "N18", "N19", "N250", "Z490", "Z491", "Z492", "Z940",
                      "Z992"),
    "malignancy": ("C0", "C1", "C2", "C30", "C31", "C32", "C33", "C34",
                   "C37", "C38", "C39", "C40", "C41", "C43", "C45", "C46",
                   "C47", "C48", "C49", "C50", "C51", "C52", "C53", "C54",
                   "C55", "C56", "C57", "C58", "C60", "C61", "C62", "C63",
                   "C64", "C65", "C66", "C67", "C68", "C69", "C70", "C71",
                   "C72", "C73", "C74", "C75", "C76", "C81", "C82", "C83",
                   "C84", "C85", "C88", "C90", "C91", "C92", "C93", "C94",
                   "C95", "C96", "C97"),
    "severe_liver_disease": ("I850", "I859", "I864", "I982", "K704", "K711",
                             "K721", "K729", "K765", "K766", "K767"),
    "metastatic_cancer": ("C77", "C78", "C79", "C80"),
    "hiv": ("B20", "B21", "B22", "B24"),
}

# Weights from the original Charlson index; used for the scalar summary score.
CHARLSON_WEIGHTS = {
    "myocardial_infarction": 1, "congestive_heart_failure": 1,
    "peripheral_vascular": 1, "cerebrovascular": 1, "dementia": 1,
    "chronic_pulmonary": 1, "rheumatic": 1, "peptic_ulcer": 1,
    "mild_liver_disease": 1, "diabetes_uncomplicated": 1,
    "diabetes_complicated": 2, "hemiplegia": 2, "renal_disease": 2,
    "malignancy": 2, "severe_liver_disease": 3, "metastatic_cancer": 6,
    "hiv": 6,
}

COMORBIDITIES = tuple(CHARLSON_WEIGHTS)

# ------------------------------------------------------- categorical binning
# MIMIC-IV race has ~30 levels, most of them sparse sub-categories. Collapsing
# to the OMB roll-up keeps the one-hot block small and the cell counts usable.
RACE_GROUPS = {
    "WHITE": "white", "BLACK": "black", "HISPANIC": "hispanic",
    "ASIAN": "asian", "AMERICAN INDIAN": "other", "NATIVE HAWAIIAN": "other",
    "OTHER": "other", "UNKNOWN": "unknown", "UNABLE TO OBTAIN": "unknown",
    "PATIENT DECLINED": "unknown", "PORTUGUESE": "other",
    "SOUTH AMERICAN": "hispanic", "MULTIPLE": "other",
}

ADMISSION_TYPES = (
    "EW EMER.", "OBSERVATION ADMIT", "URGENT", "EU OBSERVATION",
    "ELECTIVE", "SURGICAL SAME DAY ADMISSION", "DIRECT EMER.",
    "DIRECT OBSERVATION", "AMBULATORY OBSERVATION",
)

CARE_UNITS = (
    "Medical Intensive Care Unit (MICU)",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)",
    "Surgical Intensive Care Unit (SICU)",
    "Cardiac Vascular Intensive Care Unit (CVICU)",
    "Coronary Care Unit (CCU)",
    "Trauma SICU (TSICU)",
    "Neuro Intermediate", "Neuro Stepdown", "Neuro Surgical Intensive Care Unit (Neuro SICU)",
)


def race_group(value):
    """Collapse a raw MIMIC race string to its OMB roll-up."""
    if not isinstance(value, str):
        return "unknown"
    up = value.upper()
    for prefix, group in RACE_GROUPS.items():
        if up.startswith(prefix):
            return group
    return "other"


def charlson_groups(icd_code, icd_version):
    """Comorbidity groups a single ICD code belongs to (usually 0 or 1)."""
    code = str(icd_code).replace(".", "").strip().upper()
    table = CHARLSON_ICD9 if int(icd_version) == 9 else CHARLSON_ICD10
    return [g for g, prefixes in table.items() if code.startswith(prefixes)]

In [ ]:
%%writefile /kaggle/working/src/mimic_iv.py
"""Native ETL for the Kaggle `akshaybe/updated-mimic-iv` mirror of MIMIC-IV v3.1.

The mirror ships PhysioNet's directory layout verbatim:

    data/raw/mimic-iv-3.1/hosp/{patients,admissions,diagnoses_icd,labevents,...}.csv
    data/raw/mimic-iv-3.1/icu/{icustays,chartevents,inputevents,...}.csv

This module turns those tables into the single patient-level design matrix the
graph pipeline consumes. Nothing downstream knows about MIMIC schemas.

Cohort
------
Adult (>= 18) patients, **first ICU stay only** (so one row per `subject_id`,
which makes the patient-level split trivially exact), stay length at least
`min_los_hours`. Features come from a fixed observation window
`[intime, intime + obs_window_hours)`; the label is in-hospital mortality.

Leakage controls, all asserted in `build_cohort`
------------------------------------------------
1. **Window closure.** Every event used is timestamped strictly inside the
   observation window. No aggregate reaches past `intime + obs_window_hours`.
2. **Deaths inside the window are dropped.** A patient who dies at hour 12 of a
   24 h window has their outcome partly *inside* their own feature window;
   keeping them makes the task trivially easy for the wrong reason.
3. **Comorbidities come from prior admissions only.** `diagnoses_icd` holds
   billing codes assigned at *discharge*, so the current admission's codes
   encode the outcome. Only admissions whose `dischtime` precedes `intime`
   contribute. This is the single biggest leak in naive MIMIC pipelines.
4. **One stay per patient**, so no patient can straddle two splits.

Feature blocks
--------------
Blocks mirror the SBSCGM patient encoder (arXiv:2508.00615 sec. 3.2) and are
carried through to graph construction as *views*:

    demographics  age, sex, admission route, care unit, insurance, race
    comorbidity   Charlson binary groups + weighted index, from history
    labs          first-window aggregates over `hosp/labevents`
    vitals        first-window aggregates over `icu/chartevents`
    treatment     intervention exposure from inputevents/procedureevents,
                  plus urine output rate

Whichever source tables are absent on disk simply contribute no block --
`scripts/fetch_mimic.py` downloads them in tiers, and a core-only checkout still
produces a usable (if smaller) cohort.
"""

import hashlib
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

from src import mimic_concepts as mc

# Aggregates computed per numeric concept inside the observation window. `first`
# and `last` together carry the trend, which is what a static snapshot loses --
# the criticism levelled at the ICU-LoS similarity-graph work in the review.
NUMERIC_AGGS = ("first", "last", "min", "max", "mean", "count")


# ------------------------------------------------------------------ plumbing

def find_root(base="data/raw"):
    """Locate the MIMIC-IV tree under `base`. Returns a path or None."""
    for candidate in (
        os.path.join(base, "mimic-iv-3.1"),
        os.path.join(base, "mimic-iv"),
        base,
    ):
        if os.path.isdir(os.path.join(candidate, "hosp")):
            return candidate
    return None


def table_path(root, rel):
    """Path to a table, tolerating the gzipped variant. None if absent."""
    for suffix in ("", ".gz"):
        path = os.path.join(root, rel + suffix)
        if os.path.exists(path):
            return path
    return None


def has_table(root, rel):
    return table_path(root, rel) is not None


def read_table(root, rel, **kwargs):
    path = table_path(root, rel)
    if path is None:
        raise FileNotFoundError(f"{rel} not found under {root}")
    return pd.read_csv(path, **kwargs)


def _duckdb():
    """DuckDB connection, or None. Used to scan the multi-GB event tables."""
    try:
        import duckdb
    except ImportError:
        return None
    con = duckdb.connect()
    con.execute("PRAGMA threads=4")
    return con


# -------------------------------------------------------------------- cohort

def _base_cohort(root, cfg):
    """patients x admissions x icustays -> one row per included ICU stay."""
    d = cfg["data"]
    window_h = float(d["obs_window_hours"])
    min_los_h = float(d.get("min_los_hours", window_h + 4.0))

    patients = read_table(
        root, "hosp/patients.csv",
        usecols=["subject_id", "gender", "anchor_age", "anchor_year", "dod"],
    )
    admissions = read_table(
        root, "hosp/admissions.csv",
        usecols=["subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
                 "admission_type", "admission_location", "insurance",
                 "marital_status", "race", "hospital_expire_flag"],
        parse_dates=["admittime", "dischtime", "deathtime"],
    )
    stays = read_table(
        root, "icu/icustays.csv",
        usecols=["subject_id", "hadm_id", "stay_id", "first_careunit",
                 "intime", "outtime", "los"],
        parse_dates=["intime", "outtime"],
    )

    n_stays_raw = len(stays)
    # First ICU stay per patient, in time order.
    stays = stays.sort_values(["subject_id", "intime"])
    stays = stays.drop_duplicates("subject_id", keep="first")

    df = stays.merge(admissions, on=["subject_id", "hadm_id"], how="inner")
    df = df.merge(patients, on="subject_id", how="inner")

    # MIMIC-IV date-shifts each patient into an `anchor_year`; age at admission
    # is anchor_age plus the years elapsed since that anchor.
    df["age"] = df["anchor_age"] + (df["admittime"].dt.year - df["anchor_year"])

    steps = [("icu stays", n_stays_raw), ("first stay per patient", len(df))]

    df = df[df["age"] >= 18]
    steps.append(("adult", len(df)))

    df["los_hours"] = (df["outtime"] - df["intime"]).dt.total_seconds() / 3600.0
    df = df[df["los_hours"] >= min_los_h]
    steps.append((f"los >= {min_los_h:.0f}h", len(df)))

    # Leakage control 2: the outcome must fall strictly after the feature window.
    df["window_end"] = df["intime"] + pd.Timedelta(hours=window_h)
    died_in_window = df["deathtime"].notna() & (df["deathtime"] <= df["window_end"])
    df = df[~died_in_window]
    steps.append(("died after window", len(df)))

    df = df[df["dischtime"] > df["window_end"]]
    steps.append(("discharged after window", len(df)))

    df = df.reset_index(drop=True)
    df["label"] = df["hospital_expire_flag"].astype(np.int64)
    return df, steps


# ------------------------------------------------------------ feature blocks

def _demographics_block(df):
    """Always-observed context features. These anchor the similarity graph."""
    out = pd.DataFrame(index=df.index)
    out["age"] = df["age"].astype(np.float32)
    out["is_female"] = (df["gender"] == "F").astype(np.float32)
    out["los_planned_elective"] = df["admission_type"].isin(
        ("ELECTIVE", "SURGICAL SAME DAY ADMISSION")
    ).astype(np.float32)
    out["admitted_via_ed"] = df["admission_location"].fillna("").str.contains(
        "EMERGENCY"
    ).astype(np.float32)

    for adm in mc.ADMISSION_TYPES:
        key = adm.lower().replace(" ", "_").replace(".", "")
        out[f"adm_{key}"] = (df["admission_type"] == adm).astype(np.float32)
    for unit in mc.CARE_UNITS:
        key = unit.split("(")[-1].strip(")").lower().replace("/", "_").replace(" ", "_")
        out[f"unit_{key}"] = (df["first_careunit"] == unit).astype(np.float32)

    race = df["race"].map(mc.race_group)
    for group in sorted(set(mc.RACE_GROUPS.values())):
        out[f"race_{group}"] = (race == group).astype(np.float32)
    for ins in ("Medicare", "Medicaid", "Private", "Other"):
        out[f"insurance_{ins.lower()}"] = (df["insurance"] == ins).astype(np.float32)
    out["married"] = (df["marital_status"] == "MARRIED").astype(np.float32)
    return out


def _comorbidity_block(root, df):
    """Charlson groups from *prior* admissions only (leakage control 3)."""
    cols = [f"cci_{g}" for g in mc.COMORBIDITIES] + ["cci_score", "n_prior_admissions"]
    if not has_table(root, "hosp/diagnoses_icd.csv"):
        return pd.DataFrame(0.0, index=df.index, columns=cols, dtype=np.float32)

    diag = read_table(root, "hosp/diagnoses_icd.csv",
                      usecols=["subject_id", "hadm_id", "icd_code", "icd_version"])
    diag = diag[diag["subject_id"].isin(set(df["subject_id"]))]

    # Attach each diagnosis to its admission's discharge time, then keep only
    # admissions that closed before this patient's ICU window opened.
    adm_times = read_table(root, "hosp/admissions.csv",
                           usecols=["subject_id", "hadm_id", "dischtime"],
                           parse_dates=["dischtime"])
    diag = diag.merge(adm_times, on=["subject_id", "hadm_id"], how="left")
    diag = diag.merge(df[["subject_id", "intime"]], on="subject_id", how="inner")
    diag = diag[diag["dischtime"] < diag["intime"]]

    # Map every distinct (code, version) once -- there are ~2e4 of them versus
    # millions of rows.
    keys = diag[["icd_code", "icd_version"]].drop_duplicates()
    lookup = {
        (row.icd_code, row.icd_version): mc.charlson_groups(row.icd_code, row.icd_version)
        for row in keys.itertuples()
    }

    flags = {g: set() for g in mc.COMORBIDITIES}
    for subj, code, ver in zip(diag["subject_id"], diag["icd_code"], diag["icd_version"]):
        for group in lookup[(code, ver)]:
            flags[group].add(subj)

    out = pd.DataFrame(index=df.index)
    score = np.zeros(len(df), dtype=np.float32)
    for group in mc.COMORBIDITIES:
        hit = df["subject_id"].isin(flags[group]).to_numpy(dtype=np.float32)
        out[f"cci_{group}"] = hit
        score += mc.CHARLSON_WEIGHTS[group] * hit
    out["cci_score"] = score

    prior = diag.groupby("subject_id")["hadm_id"].nunique()
    out["n_prior_admissions"] = (
        df["subject_id"].map(prior).fillna(0.0).to_numpy(dtype=np.float32)
    )
    return out


def _window_sql(source, id_col, time_col, value_col, item_col, itemids):
    """SQL that aggregates one event table inside each stay's window."""
    ids = ", ".join(str(i) for i in itemids)
    return f"""
        SELECT w.row_id, e.{item_col} AS itemid,
               min_by(e.{value_col}, e.{time_col}) AS v_first,
               max_by(e.{value_col}, e.{time_col}) AS v_last,
               min(e.{value_col})   AS v_min,
               max(e.{value_col})   AS v_max,
               avg(e.{value_col})   AS v_mean,
               count(e.{value_col}) AS v_count
        FROM read_csv_auto('{source}', ignore_errors=true) e
        JOIN win w ON e.{id_col} = w.{id_col}
        WHERE e.{item_col} IN ({ids})
          AND e.{value_col} IS NOT NULL
          AND e.{time_col} >= w.intime
          AND e.{time_col} <  w.window_end
        GROUP BY w.row_id, e.{item_col}
    """


def _aggregate_events(root, df, rel, id_col, time_col, value_col, items,
                      prefix, chunksize=4_000_000, verbose=True):
    """Window-aggregate a large event table into per-concept feature columns.

    Uses DuckDB when it is importable (it scans the 18-42 GB CSVs with predicate
    pushdown and constant memory); otherwise falls back to a chunked pandas
    scan, which is slower but has no extra dependency.
    """
    concepts = sorted({name for name, *_ in items.values()})
    ranges = {name: (lo, hi) for name, _, lo, hi in items.values()}
    columns = [f"{prefix}_{c}_{a}" for c in concepts for a in NUMERIC_AGGS]

    path = table_path(root, rel)
    if path is None:
        if verbose:
            print(f"  [{prefix}] {rel} absent -- block skipped")
        return pd.DataFrame(np.nan, index=df.index, columns=columns, dtype=np.float32)

    win = df[[id_col, "intime", "window_end"]].copy()
    win["row_id"] = np.arange(len(df))

    con = _duckdb()
    if con is not None:
        if verbose:
            print(f"  [{prefix}] duckdb scan of {rel} ...", flush=True)
        con.register("win", win)
        agg = con.execute(
            _window_sql(path, id_col, time_col, value_col, "itemid", sorted(items))
        ).fetch_df()
        con.close()
    else:
        agg = _aggregate_events_pandas(path, win, id_col, time_col, value_col,
                                       items, chunksize, verbose, prefix)

    out = pd.DataFrame(np.nan, index=df.index, columns=columns, dtype=np.float32)
    if len(agg) == 0:
        return out

    agg["concept"] = agg["itemid"].map({k: v[0] for k, v in items.items()})
    # One itemid per concept per stay is the common case, but pressures and
    # glucose have several sources; combine them the way each aggregate implies.
    grouped = agg.groupby(["row_id", "concept"]).agg(
        v_first=("v_first", "first"), v_last=("v_last", "last"),
        v_min=("v_min", "min"), v_max=("v_max", "max"),
        v_mean=("v_mean", "mean"), v_count=("v_count", "sum"),
    ).reset_index()

    for concept, block in grouped.groupby("concept"):
        lo, hi = ranges[concept]
        rows = block["row_id"].to_numpy()
        for agg_name in NUMERIC_AGGS:
            vals = block[f"v_{agg_name}"].to_numpy(dtype=np.float64)
            if agg_name != "count":
                vals = np.where((vals < lo) | (vals > hi), np.nan, vals)
            # `out` is float32; assigning float64 into it is deprecated in
            # pandas and becomes an error in a future version.
            out.iloc[rows, out.columns.get_loc(f"{prefix}_{concept}_{agg_name}")] = \
                vals.astype(np.float32)

    # A concept that was never measured has count 0, not "missing count".
    for concept in concepts:
        col = f"{prefix}_{concept}_count"
        out[col] = out[col].fillna(0.0)
    return out.astype(np.float32)


def _aggregate_events_pandas(path, win, id_col, time_col, value_col, items,
                             chunksize, verbose, prefix):
    """Chunked fallback for `_aggregate_events` when DuckDB is unavailable."""
    keep_ids = set(win[id_col])
    bounds = win.set_index(id_col)[["intime", "window_end", "row_id"]]
    parts = []
    reader = pd.read_csv(
        path, usecols=[id_col, time_col, value_col, "itemid"],
        parse_dates=[time_col], chunksize=chunksize,
        dtype={value_col: np.float64}, on_bad_lines="skip",
    )
    for i, chunk in enumerate(reader):
        chunk = chunk[chunk["itemid"].isin(items) & chunk[id_col].isin(keep_ids)]
        chunk = chunk.dropna(subset=[value_col])
        if chunk.empty:
            continue
        chunk = chunk.join(bounds, on=id_col)
        chunk = chunk[(chunk[time_col] >= chunk["intime"])
                      & (chunk[time_col] < chunk["window_end"])]
        if not chunk.empty:
            parts.append(chunk[["row_id", "itemid", time_col, value_col]])
        if verbose and i % 10 == 0:
            print(f"  [{prefix}] chunk {i}, {sum(len(p) for p in parts):,} rows kept",
                  flush=True)

    if not parts:
        return pd.DataFrame(columns=["row_id", "itemid", "v_first", "v_last",
                                     "v_min", "v_max", "v_mean", "v_count"])
    events = pd.concat(parts).sort_values(time_col)
    return events.groupby(["row_id", "itemid"])[value_col].agg(
        v_first="first", v_last="last", v_min="min", v_max="max",
        v_mean="mean", v_count="count",
    ).reset_index()


def _treatment_block(root, df, verbose=True):
    """Binary intervention exposure inside the window, plus urine output rate.

    These are the "interventions and medications" features of the SBSCGM
    encoder: whether a patient was on pressors, ventilated, or dialysed in the
    first window is both highly prognostic and cheap to extract.
    """
    names = sorted(set(mc.INPUT_ITEMS.values()) | set(mc.PROC_ITEMS.values()))
    columns = [f"tx_{n}" for n in names] + ["tx_any_vasopressor", "tx_n_interventions",
                                            "tx_urine_ml_per_hr"]
    out = pd.DataFrame(0.0, index=df.index, columns=columns, dtype=np.float32)

    win = df[["stay_id", "intime", "window_end"]].copy()
    win["row_id"] = np.arange(len(df))
    window_h = (df["window_end"] - df["intime"]).dt.total_seconds().to_numpy() / 3600.0

    sources = [
        ("icu/inputevents.csv", "starttime", mc.INPUT_ITEMS),
        ("icu/procedureevents.csv", "starttime", mc.PROC_ITEMS),
    ]
    for rel, time_col, mapping in sources:
        path = table_path(root, rel)
        if path is None:
            if verbose:
                print(f"  [tx] {rel} absent -- block partially skipped")
            continue
        hits = _exposure_rows(path, win, time_col, sorted(mapping), verbose, rel)
        for itemid, rows in hits.items():
            col = f"tx_{mapping[itemid]}"
            out.iloc[list(rows), out.columns.get_loc(col)] = 1.0

    out["tx_any_vasopressor"] = out[
        [f"tx_{v}" for v in mc.VASOPRESSORS if f"tx_{v}" in out]
    ].max(axis=1)
    out["tx_n_interventions"] = out[[f"tx_{n}" for n in names]].sum(axis=1)

    urine_path = table_path(root, "icu/outputevents.csv")
    if urine_path is not None:
        totals = _urine_totals(urine_path, win, verbose)
        rate = np.zeros(len(df), dtype=np.float32)
        rate[list(totals)] = np.array(list(totals.values()), dtype=np.float32)
        rate = rate / np.maximum(window_h, 1e-6)
        # Irrigant subtraction can drive a stay negative; both that and a
        # supraphysiologic rate mean "not measurable", which is missing, not 0.
        out["tx_urine_ml_per_hr"] = np.where(
            (rate < 0) | (rate > mc.URINE_RATE_MAX_ML_PER_HR), np.nan, rate)
    else:
        out["tx_urine_ml_per_hr"] = np.nan
    return out


def _exposure_rows(path, win, time_col, itemids, verbose, rel):
    """{itemid: set(row_id)} for events starting inside each stay's window."""
    con = _duckdb()
    ids = ", ".join(str(i) for i in itemids)
    if con is not None:
        if verbose:
            print(f"  [tx] duckdb scan of {rel} ...", flush=True)
        con.register("win", win)
        res = con.execute(f"""
            SELECT DISTINCT e.itemid, w.row_id
            FROM read_csv_auto('{path}', ignore_errors=true) e
            JOIN win w ON e.stay_id = w.stay_id
            WHERE e.itemid IN ({ids})
              AND e.{time_col} >= w.intime AND e.{time_col} < w.window_end
        """).fetch_df()
        con.close()
    else:
        bounds = win.set_index("stay_id")[["intime", "window_end", "row_id"]]
        keep, parts = set(win["stay_id"]), []
        for chunk in pd.read_csv(path, usecols=["stay_id", "itemid", time_col],
                                 parse_dates=[time_col], chunksize=2_000_000,
                                 on_bad_lines="skip"):
            chunk = chunk[chunk["itemid"].isin(itemids) & chunk["stay_id"].isin(keep)]
            if chunk.empty:
                continue
            chunk = chunk.join(bounds, on="stay_id")
            chunk = chunk[(chunk[time_col] >= chunk["intime"])
                          & (chunk[time_col] < chunk["window_end"])]
            parts.append(chunk[["itemid", "row_id"]])
        res = pd.concat(parts).drop_duplicates() if parts else pd.DataFrame(
            columns=["itemid", "row_id"])

    return {int(i): set(b["row_id"]) for i, b in res.groupby("itemid")}


def _urine_totals(path, win, verbose):
    """{row_id: total urine volume (mL)} inside each stay's window."""
    all_items = mc.OUTPUT_URINE_ITEMS + mc.OUTPUT_URINE_NEGATIVE_ITEMS
    ids = ", ".join(str(i) for i in all_items)
    neg_ids = ", ".join(str(i) for i in mc.OUTPUT_URINE_NEGATIVE_ITEMS)
    con = _duckdb()
    if con is not None:
        if verbose:
            print("  [tx] duckdb scan of outputevents ...", flush=True)
        con.register("win", win)
        res = con.execute(f"""
            SELECT w.row_id,
                   sum(CASE WHEN e.itemid IN ({neg_ids}) THEN -e.value
                            ELSE e.value END) AS total
            FROM read_csv_auto('{path}', ignore_errors=true) e
            JOIN win w ON e.stay_id = w.stay_id
            WHERE e.itemid IN ({ids}) AND e.value IS NOT NULL
              AND e.value BETWEEN 0 AND {mc.URINE_EVENT_MAX_ML}
              AND e.charttime >= w.intime AND e.charttime < w.window_end
            GROUP BY w.row_id
        """).fetch_df()
        con.close()
    else:
        bounds = win.set_index("stay_id")[["intime", "window_end", "row_id"]]
        keep, parts = set(win["stay_id"]), []
        for chunk in pd.read_csv(path, usecols=["stay_id", "itemid", "charttime", "value"],
                                 parse_dates=["charttime"], chunksize=2_000_000,
                                 on_bad_lines="skip"):
            chunk = chunk[chunk["itemid"].isin(all_items)
                          & chunk["stay_id"].isin(keep)]
            chunk = chunk[chunk["value"].between(0, mc.URINE_EVENT_MAX_ML)]
            if chunk.empty:
                continue
            chunk = chunk.join(bounds, on="stay_id")
            chunk = chunk[(chunk["charttime"] >= chunk["intime"])
                          & (chunk["charttime"] < chunk["window_end"])]
            chunk["value"] = np.where(
                chunk["itemid"].isin(mc.OUTPUT_URINE_NEGATIVE_ITEMS),
                -chunk["value"], chunk["value"])
            parts.append(chunk[["row_id", "value"]])
        res = (pd.concat(parts).groupby("row_id")["value"].sum().reset_index(name="total")
               if parts else pd.DataFrame(columns=["row_id", "total"]))
    return dict(zip(res["row_id"].astype(int), res["total"].astype(float)))


# ------------------------------------------------------------- derived signal

def _severity(df, blocks):
    """Severity from the observation window only -- no outcome, no full-stay info.

    This exists because `_criticalness` weights the death flag at 1.2 against
    two standardised terms, so it correlates with mortality at AUROC ~0.97. That
    is fine for an auxiliary *training* target, but it makes it useless as an
    evaluation stratum: "top decile of criticalness" is very nearly "the
    patients who died", which is the same degeneracy `high_risk_analysis` was
    fixed to avoid (HANDOVER 6.1).

    A mini-SOFA instead: organ-support exposure plus physiologic derangement,
    all measured inside `[intime, intime + obs_window)`. Every component is
    already a model input, which is fine -- the stratum has to be independent of
    the model's *decisions* and of the *outcome*, not of the features.

    Components are robust-z-scored (median / IQR) so a single wild lab does not
    dominate, oriented so larger is sicker, and averaged over whichever are
    available.
    """
    cols = pd.concat([b for b in blocks.values() if b is not None and b.shape[1]],
                     axis=1) if blocks else pd.DataFrame(index=df.index)

    # (column, +1 if high is sicker, -1 if low is sicker)
    COMPONENTS = [
        ("tx_any_vasopressor", +1), ("tx_invasive_ventilation", +1),
        ("tx_crrt", +1), ("tx_n_interventions", +1),
        ("lab_lactate_max", +1), ("lab_creatinine_max", +1), ("lab_bun_max", +1),
        ("vit_gcs_motor_last", -1), ("vit_gcs_verbal_last", -1),
        ("vit_sbp_min", -1), ("vit_spo2_min", -1), ("vit_resp_rate_max", +1),
    ]

    parts, used = [], []
    for name, sign in COMPONENTS:
        if name not in cols.columns:
            continue
        v = cols[name].to_numpy(dtype=np.float64)
        if not np.isfinite(v).any():
            continue
        med = np.nanmedian(v)
        q1, q3 = np.nanpercentile(v[np.isfinite(v)], [25, 75])
        scale = max(q3 - q1, 1e-6)
        z = sign * (np.where(np.isfinite(v), v, med) - med) / scale
        parts.append(np.clip(z, -4.0, 4.0))
        used.append(name)

    if not parts:
        return np.zeros(len(df), dtype=np.float32), []

    raw = np.mean(parts, axis=0)
    lo, hi = np.percentile(raw, [1, 99])
    score = np.clip((raw - lo) / max(hi - lo, 1e-8), 0.0, 1.0)
    return score.astype(np.float32), used


def _criticalness(df, treatment):
    """Continuous severity proxy used as the auxiliary regression target.

    Follows the SBSCGM multi-task setup (arXiv:2508.00615 sec. 3.4): a
    normalised blend of intervention burden, ICU length of stay, and discharge
    disposition. It is a *label*, derived from the full stay, and is only ever
    read on training nodes inside the loss -- never a model input.

    Note the death flag carries weight 1.2 against two standardised terms, so
    this correlates with mortality at AUROC ~0.97 on the MIMIC cohort. That is
    acceptable for a training target but disqualifies it as an evaluation
    stratum -- use `_severity` for that.
    """
    los = np.log1p(df["los_hours"].to_numpy(dtype=np.float64))
    los = (los - los.mean()) / (los.std() + 1e-8)
    burden = treatment["tx_n_interventions"].to_numpy(dtype=np.float64)
    burden = (burden - burden.mean()) / (burden.std() + 1e-8)
    died = df["label"].to_numpy(dtype=np.float64)

    raw = 0.35 * los + 0.35 * burden + 1.2 * died
    lo, hi = np.percentile(raw, [1, 99])
    return np.clip((raw - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)


# ----------------------------------------------------------------- top level

# Bump when the *block-building* semantics change (itemid sets, aggregates,
# window logic). Expensive: it invalidates the 18 GB and 42 GB scans.
_BLOCK_VERSION = 2
# Bump when *assembly* semantics change (cohort filters, packing, derived
# columns). Cheap: blocks are reused, only the assembled table is rebuilt.
_COHORT_VERSION = 4

_ROW_KEYS = ("source", "task", "obs_window_hours", "min_los_hours",
             "max_patients", "cohort_seed")


def row_signature(cfg):
    """Hash of the fields that decide *which patients* are in the cohort.

    Feature blocks are cached against this rather than against
    `cohort_signature`, so enabling one more block reuses the multi-GB scans
    already done for the others instead of invalidating them.
    """
    d = cfg["data"]
    payload = {k: d.get(k) for k in _ROW_KEYS}
    payload["_v"] = _BLOCK_VERSION
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha1(blob.encode()).hexdigest()[:10]


def cohort_signature(cfg):
    """Hash of only the fields that change the extracted cohort."""
    d = cfg["data"]
    keys = _ROW_KEYS + ("blocks", "require_nonempty_blocks")
    payload = {k: d.get(k) for k in keys}
    payload["_v"] = (_BLOCK_VERSION, _COHORT_VERSION)
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha1(blob.encode()).hexdigest()[:10]


def _block_present(block):
    """Boolean mask: did this patient have *any* measurement in this block?

    Not `notna().any()`. `_aggregate_events` fills the `_count` columns with
    zero on purpose -- a concept that was never measured has count 0, not a
    missing count -- so every row is non-null in those columns and a naive
    null check calls every patient covered. The count columns are precisely
    the ones that answer the question, when they exist.
    """
    counts = [c for c in block.columns if c.endswith("_count")]
    if counts:
        return (block[counts].fillna(0).to_numpy() > 0).any(axis=1)
    return block.notna().any(axis=1).to_numpy()


def _drop_uncovered(df, blocks, required, verbose=True):
    """Drop patients for whom a required feature block is entirely absent.

    Public MIMIC mirrors are not always complete. The Kaggle
    `akshaybe/updated-mimic-iv` copy ships a `chartevents.csv` that is truncated
    mid-record and covers only the first ~38% of ICU stays in `subject_id`
    order, so most patients have no vitals at all.

    Leaving them in is worse than dropping them. `impute_and_scale` appends a
    missingness indicator per column *because* missingness in the ICU is
    informative -- a lab not drawn is a clinical decision. When the missingness
    is instead an artefact of a truncated download, those indicators encode
    "this patient's subject_id sorts below the cut point", and the model learns
    the defect as signal.

    `subject_id` is randomly assigned in MIMIC-IV, so the surviving cohort is
    not clinically selected -- but that is a property of this particular
    truncation, and is asserted rather than assumed by the caller.
    """
    if not required:
        return df, blocks, np.ones(len(df), dtype=bool)

    keep = np.ones(len(df), dtype=bool)
    for name in required:
        block = blocks.get(name)
        if block is None:
            raise ValueError(f"require_nonempty_blocks names {name!r}, which was not built")
        present = _block_present(block)
        if verbose:
            print(f"  [{name}] present for {present.sum():,} / {len(present):,} "
                  f"patients ({100 * present.mean():.1f}%)")
        keep &= present

    if keep.all():
        return df, blocks, keep
    if verbose:
        print(f"  dropped {(~keep).sum():,} patients lacking a required block "
              f"-> {keep.sum():,} remain")
    df = df[keep].reset_index(drop=True)
    blocks = {k: v[keep].reset_index(drop=True) for k, v in blocks.items()}
    return df, blocks, keep


def _cached_block(name, sig, cache_dir, build, force=False, verbose=True):
    """Build one feature block, memoised on disk.

    The `labs` and `vitals` blocks each cost a full scan of a multi-GB event
    table. On a hosted kernel with a wall-clock session limit, losing those to
    an unrelated failure later in the pipeline is expensive, so every block is
    written the moment it is built and reused on the next run.
    """
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"block_{sig}_{name}.parquet")
    if os.path.exists(path) and not force:
        block = pd.read_parquet(path)
        if verbose:
            print(f"  [{name}] cached ({block.shape[1]} cols)")
        return block
    t0 = time.time()
    block = build()
    block.to_parquet(path, index=False)
    if verbose:
        print(f"  [{name}] built {block.shape[1]} cols in {time.time() - t0:.1f}s "
              f"-> {path}")
    return block


def build_cohort(cfg, verbose=True):
    """Extract (or load a cached) MIMIC-IV cohort.

    Returns `(X, y, feature_names, meta)` with the same contract as the mock
    generator. `X` keeps NaN for genuinely unmeasured values -- imputation
    happens later, fit on the training split only.
    """
    d = cfg["data"]
    raw_dir = d.get("raw_dir", "data/raw")
    cache_dir = d.get("cache_dir", "data/processed")
    root = find_root(raw_dir)
    if root is None:
        raise FileNotFoundError(
            f"no MIMIC-IV tree under {raw_dir}. Run:\n"
            f"    python scripts/fetch_mimic.py --tier core --tier events"
        )

    sig = cohort_signature(cfg)
    row_sig = row_signature(cfg)
    cache = os.path.join(cache_dir, f"cohort_{sig}.csv.gz")
    if os.path.exists(cache) and not d.get("force_rebuild", False):
        if verbose:
            print(f"[mimic] cached cohort {cache}")
        return _unpack(pd.read_csv(cache), cfg)

    if verbose:
        print(f"[mimic] building cohort from {root}")
    df, steps = _base_cohort(root, cfg)
    if verbose:
        for name, n in steps:
            print(f"  {name:28s} {n:>8,}")

    max_patients = d.get("max_patients")
    if max_patients and len(df) > max_patients:
        # Subsample *patients*, stratified on the label so prevalence survives.
        rng = np.random.default_rng(d.get("cohort_seed", cfg["seed"]))
        keep = np.concatenate([
            rng.permutation(np.flatnonzero(df["label"].to_numpy() == cls))[
                : int(round(max_patients * (df["label"] == cls).mean()))
            ]
            for cls in (0, 1)
        ])
        df = df.iloc[np.sort(keep)].reset_index(drop=True)
        if verbose:
            print(f"  {'subsampled':28s} {len(df):>8,}")

    enabled = set(d.get("blocks") or
                  ["demographics", "comorbidity", "labs", "vitals", "treatment"])
    builders = {
        "demographics": lambda: _demographics_block(df),
        "comorbidity": lambda: _comorbidity_block(root, df),
        "labs": lambda: _aggregate_events(
            root, df, "hosp/labevents.csv", "hadm_id", "charttime", "valuenum",
            mc.LAB_ITEMS, "lab", verbose=verbose),
        "vitals": lambda: _aggregate_events(
            root, df, "icu/chartevents.csv", "stay_id", "charttime", "valuenum",
            mc.CHART_ITEMS, "vit", verbose=verbose),
        "treatment": lambda: _treatment_block(root, df, verbose=verbose),
    }
    blocks, views = {}, {}
    for name in ("demographics", "comorbidity", "labs", "vitals", "treatment"):
        if name in enabled:
            blocks[name] = _cached_block(name, row_sig, cache_dir, builders[name],
                                         force=d.get("force_rebuild", False),
                                         verbose=verbose)

    # Drop blocks that came back entirely empty (source table not downloaded).
    blocks = {k: v for k, v in blocks.items()
              if v.shape[1] and not v.isna().all().all()}

    # Then drop patients for whom a required block is missing entirely, which
    # is how a partial mirror shows up (see `_drop_uncovered`).
    df, blocks, _kept = _drop_uncovered(
        df, blocks, [b for b in (d.get("require_nonempty_blocks") or []) if b in blocks],
        verbose=verbose)

    cursor = 0
    for name, block in blocks.items():
        views[name] = list(range(cursor, cursor + block.shape[1]))
        cursor += block.shape[1]

    features = pd.concat(list(blocks.values()), axis=1)
    crit = _criticalness(df, blocks.get("treatment",
                                        pd.DataFrame({"tx_n_interventions": np.zeros(len(df))})))
    sev, sev_used = _severity(df, blocks)
    if verbose:
        print(f"  [severity] {len(sev_used)} components: {', '.join(sev_used)}")

    packed = features.copy()
    packed.insert(0, "subject_id", df["subject_id"].to_numpy())
    packed.insert(1, "hadm_id", df["hadm_id"].to_numpy())
    packed.insert(2, "stay_id", df["stay_id"].to_numpy())
    packed.insert(3, "label", df["label"].to_numpy())
    packed.insert(4, "criticalness", crit)
    packed.insert(5, "los_hours", df["los_hours"].to_numpy())
    packed.insert(6, "severity", sev)

    os.makedirs(cache_dir, exist_ok=True)
    packed.to_csv(cache, index=False)
    with open(os.path.join(cache_dir, f"cohort_{sig}.views.json"), "w") as f:
        json.dump({"views": views, "steps": steps}, f, indent=2, default=str)
    if verbose:
        print(f"[mimic] wrote {cache}  ({len(packed):,} patients, "
              f"{features.shape[1]} features, prevalence "
              f"{packed['label'].mean():.3f})")
    return _unpack(packed, cfg)


_ID_COLS = ("subject_id", "hadm_id", "stay_id", "label", "criticalness",
            "los_hours", "severity")


def _unpack(packed, cfg):
    """Split the cached table into the (X, y, names, meta) contract."""
    sig = cohort_signature(cfg)
    view_path = os.path.join(cfg["data"].get("cache_dir", "data/processed"),
                             f"cohort_{sig}.views.json")
    views = {}
    if os.path.exists(view_path):
        with open(view_path) as f:
            views = {k: list(map(int, v)) for k, v in json.load(f)["views"].items()}

    names = [c for c in packed.columns if c not in _ID_COLS]
    X = packed[names].to_numpy(dtype=np.float32)
    y = packed["label"].to_numpy(dtype=np.int64)

    # Columns that are already 0/1 drive the Jaccard half of the hybrid
    # similarity (arXiv:2508.00615 sec. 3.1); everything else is continuous.
    finite = np.where(np.isfinite(X), X, 0.0)
    binary = np.all((finite == 0.0) | (finite == 1.0), axis=0)

    meta = {
        "subject_id": packed["subject_id"].to_numpy(),
        "hadm_id": packed["hadm_id"].to_numpy(),
        "stay_id": packed["stay_id"].to_numpy(),
        "criticalness": packed["criticalness"].to_numpy(dtype=np.float32),
        "severity": packed["severity"].to_numpy(dtype=np.float32),
        # Time axis for the concordance index. ICU length of stay with
        # in-hospital death as the event -- survivors censored at discharge.
        "los_hours": packed["los_hours"].to_numpy(dtype=np.float32),
        "views": views,
        "binary_cols": np.flatnonzero(binary),
    }
    if len(y) and y.sum() == 0:
        warnings.warn("cohort has no positive labels -- check the window filters")
    return X, y, names, meta

In [ ]:
%%writefile /kaggle/working/src/graph_v2.py
"""Graph v2: label-aware patient-graph models (notebooks 12-13).

Shared by the Kaggle notebooks via `scripts/nbmake.py`'s `#%% embed` marker.
Everything here is torch/numpy only (no xgboost), device-agnostic.

Pieces
------
- `prep_views`       per-view preprocessing (iqr / pct / quantile) + optional
                     per-feature missingness indicators, fit on train rows only.
- `Encoder`, `EncoderOnly`
                     multi-view MLP encoder -> h; graph-free exit.
- `Retrieval`        TabR-style label-aware retrieval (G1): attends over the m
                     nearest pool patients (self excluded) in a learned key space.
- `fit_minibatch`, `predict_rows`
                     generic mini-batch trainer (AdamW, early stopping on a
                     caller-supplied validation AUROC) + batched inference.
- `crossfit`         K-fold cross-fitting: out-of-fold logits for every training
                     row, fold-averaged logits for held-out rows.
- `cs_features`      leak-free Correct & Smooth features over a kNN graph whose
                     nodes are training patients: 1-hop / 2-hop neighbour OOF
                     residuals and labels, with the query's own label excluded
                     on every path (self-edges removed, 2-hop self-returns
                     subtracted).

Leakage invariants (tested in tests/test_graph_v2.py):
  * a training row's OOF logit never comes from a model that saw its label;
  * a training row's C&S features never include its own residual/label;
  * held-out rows only ever read training rows' labels/residuals.
"""

import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer


def auroc(y, p):
    return float(roc_auc_score(y, p))


# ------------------------------------------------------------ preprocessing

def prep_views(X, views, tr, kind="iqr", indicators=True, return_cont=False):
    """-> (Xp [n, D] float32, slices {view: (start, end)}). Stats fit on `tr` rows.
    With return_cont=True also returns {view: bool mask over that view's
    columns} marking continuous (non-binary, non-indicator) columns -- the
    ones `Encoder(plr=...)` embeds."""
    blocks, slices, start, conts = [], {}, 0, {}
    for v, idx in views.items():
        x = X[:, idx].astype(np.float64)
        miss = np.isnan(x)
        allmiss = miss[tr].all(0)
        x, miss = x[:, ~allmiss], miss[:, ~allmiss]
        obs_tr = np.where(miss[tr], np.nan, x[tr])
        binary = np.array([np.isin(c[~np.isnan(c)], (0.0, 1.0)).all() for c in obs_tr.T])
        med = np.nan_to_num(np.nanmedian(obs_tr, 0))
        z = np.zeros_like(x)
        cont = ~binary
        if kind == "quantile":
            qt = QuantileTransformer(n_quantiles=256, output_distribution="normal",
                                     subsample=10**9, random_state=0)
            qt.fit(obs_tr[:, cont])
            z[:, cont] = np.nan_to_num(qt.transform(x[:, cont]), nan=0.0)
        else:
            q1, q3 = np.nanpercentile(obs_tr, 25, 0), np.nanpercentile(obs_tr, 75, 0)
            iqr = np.where(np.isnan(q3 - q1) | (q3 - q1 < 1e-6), 1.0, q3 - q1)
            if kind == "iqr":
                lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            else:  # pct
                lo, hi = np.nanpercentile(obs_tr, 0.5, 0), np.nanpercentile(obs_tr, 99.5, 0)
            xc = np.clip(np.where(miss, med, x), lo, hi)
            z[:, cont] = ((xc - med) / iqr)[:, cont]
        xb = np.where(miss, med, x)[:, binary]
        z[:, binary] = xb - xb[tr].mean(0)
        parts = [z]
        if indicators:
            keep = miss[tr].any(0)
            parts += [miss[:, keep].astype(np.float64), 1.0 - miss.mean(1, keepdims=True)]
        zz = np.concatenate(parts, 1).astype(np.float32)
        conts[v] = np.concatenate([cont, np.zeros(zz.shape[1] - len(cont), dtype=bool)])
        slices[v] = (start, start + zz.shape[1])
        start += zz.shape[1]
        blocks.append(zz)
    if return_cont:
        return np.concatenate(blocks, 1), slices, conts
    return np.concatenate(blocks, 1), slices


# ------------------------------------------------------------------ models

class PLR(nn.Module):
    """Periodic -> Linear -> ReLU numeric embedding (Gorishniy et al. 2022),
    "lite" variant: per-feature learnable frequencies, one Linear shared by
    all features. [B, n] -> [B, n * e]."""

    def __init__(s, n_feat, n_freq=16, sigma=0.1, e=8):
        super().__init__()
        s.c = nn.Parameter(torch.randn(n_feat, n_freq) * sigma)
        s.lin = nn.Linear(2 * n_freq, e)

    def forward(s, x):
        v = 2 * math.pi * s.c[None] * x[..., None]
        return F.relu(s.lin(torch.cat([torch.cos(v), torch.sin(v)], -1))).flatten(1)


class ViewEnc(nn.Module):
    def __init__(s, i, o, hid, p):
        super().__init__()
        s.fc1, s.fc2 = nn.Linear(i, hid), nn.Linear(hid, o)
        s.drop, s.ln = nn.Dropout(p), nn.LayerNorm(o)

    def forward(s, x):
        return s.ln(s.fc2(s.drop(F.gelu(s.fc1(x)))))


class Encoder(nn.Module):
    """Multi-view encoder: one MLP per view, residual fusion -> h [n, d]."""

    def __init__(s, slices, d=128, hid=128, vout=32, p=0.2, cont=None, plr=None):
        super().__init__()
        s.slices = dict(slices)
        s.plr = nn.ModuleDict()
        s.cont_idx, s.rest_idx = {}, {}
        in_dims = {}
        for v, (a, b) in s.slices.items():
            if plr and cont is not None and cont[v].any():
                ci = torch.as_tensor(np.flatnonzero(cont[v]) + a)
                ri = torch.as_tensor(np.flatnonzero(~cont[v]) + a)
                s.register_buffer(f"ci_{v}", ci); s.register_buffer(f"ri_{v}", ri)
                s.plr[v] = PLR(len(ci), **plr)
                in_dims[v] = len(ci) * plr.get("e", 8) + len(ri)
            else:
                in_dims[v] = b - a
        s.enc = nn.ModuleDict({v: ViewEnc(in_dims[v], vout, hid, p) for v in s.slices})
        cat = vout * len(s.slices)
        s.skip = nn.Linear(cat, d)
        s.fuse = nn.Sequential(nn.Linear(cat, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d)

    def _view_in(s, x, v, a, b):
        if v not in s.plr:
            return x[:, a:b]
        ci, ri = getattr(s, f"ci_{v}"), getattr(s, f"ri_{v}")
        return torch.cat([s.plr[v](x[:, ci]), x[:, ri]], 1)

    def forward(s, x):
        z = torch.cat([s.enc[v](s._view_in(x, v, a, b)) for v, (a, b) in s.slices.items()], 1)
        return s.ln(s.skip(z) + s.fuse(z))


def head(i, p):
    return nn.Sequential(nn.Linear(i, i), nn.GELU(), nn.Dropout(p), nn.Linear(i, 1))


class EncoderOnly(nn.Module):
    def __init__(s, slices, d=128, p=0.2, hid=128, cont=None, plr=None):
        super().__init__()
        s.enc, s.h0 = Encoder(slices, d=d, hid=hid, p=p, cont=cont, plr=plr), head(d, p)

    def forward(s, x):
        return [s.h0(s.enc(x)).squeeze(-1)]


class Retrieval(nn.Module):
    """G1. For a query batch: keys of the whole pool are computed without
    gradient, the m nearest (L2) are picked -- the query itself excluded when
    it is a pool member (`q_pool_pos`) -- and only those are re-encoded with
    gradient. Neighbour value = LabelEmb(y_j) + T(k_q - k_j), softmax(-dist)
    weighted. Exit 0 = graph-free head on h, exit 1 = after retrieval."""

    def __init__(s, slices, d=128, m=32, p=0.2, ctx_drop=0.2, hid=128, cont=None, plr=None):
        super().__init__()
        s.enc, s.m = Encoder(slices, d=d, hid=hid, p=p, cont=cont, plr=plr), m
        s.K = nn.Linear(d, d)
        s.lab = nn.Embedding(2, d)
        s.T = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(p), nn.Linear(d, d, bias=False))
        s.block = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2 * d), nn.ReLU(), nn.Dropout(p),
                                nn.Linear(2 * d, d))
        s.ctx_drop = nn.Dropout(ctx_drop)
        s.h0, s.h1 = head(d, p), head(d, p)

    def forward(s, xq, q_pool_pos, x_pool, y_pool):
        hq = s.enc(xq)
        kq = s.K(hq)
        with torch.no_grad():
            kp = torch.cat([s.K(s.enc(x_pool[i:i + 4096])) for i in range(0, len(x_pool), 4096)])
            dist = torch.cdist(kq.detach(), kp) ** 2
            if q_pool_pos is not None:
                own = q_pool_pos >= 0
                rows = torch.arange(len(xq), device=xq.device)[own]
                dist[rows, q_pool_pos[own]] = float("inf")
            nbr = dist.topk(min(s.m, len(x_pool) - 1), dim=1, largest=False).indices
        kn = s.K(s.enc(x_pool[nbr.reshape(-1)])).reshape(len(xq), nbr.shape[1], -1)
        w = s.ctx_drop(torch.softmax(-((kq[:, None, :] - kn) ** 2).sum(-1), dim=1))
        v = s.lab(y_pool[nbr]) + s.T(kq[:, None, :] - kn)
        h1 = hq + (w[..., None] * v).sum(1)
        h1 = h1 + s.block(h1)
        return [s.h0(hq).squeeze(-1), s.h1(h1).squeeze(-1)]


# ---------------------------------------------------------------- training

def fit_minibatch(model, forward_fn, tr_idx, y_t, val_fn, lr=1e-3, wd=1e-4, bs=512,
                  max_ep=300, patience=20, seed=0, restore_best=True):
    """`forward_fn(idx, train=True)` -> list of exit logits for rows idx;
    loss = mean BCE over exits. `val_fn()` -> validation AUROC of the LAST
    exit; best epoch restored (unless restore_best=False)."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    rng = np.random.default_rng(seed)
    best, best_state, bad, best_ep, t0 = -1.0, None, 0, 0, time.time()
    ep = -1
    for ep in range(max_ep):
        model.train()
        perm = rng.permutation(tr_idx)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            if len(b) < 2:
                continue
            outs = forward_fn(b, train=True)
            loss = sum(F.binary_cross_entropy_with_logits(o, y_t[b]) for o in outs) / len(outs)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        v = val_fn()
        if v > best:
            best, bad, best_ep = v, 0, ep
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if restore_best and best_state is not None:
        model.load_state_dict(best_state)
    return {"val_auroc": best, "best_epoch": best_ep, "epochs_run": ep + 1,
            "train_s": time.time() - t0}


@torch.no_grad()
def predict_rows(forward_fn, model, idx, bs=2048):
    model.eval()
    outs = None
    for s in range(0, len(idx), bs):
        o = [t.float().cpu().numpy() for t in forward_fn(idx[s:s + bs], train=False)]
        outs = o if outs is None else [np.concatenate([a, b]) for a, b in zip(outs, o)]
    return outs


def stratified_folds(y_tr, n_folds, seed):
    """Fold id per training row (positions into tr_idx), class-stratified."""
    rng = np.random.default_rng(seed)
    fold = np.empty(len(y_tr), dtype=int)
    for c in np.unique(y_tr):
        idx = rng.permutation(np.flatnonzero(y_tr == c))
        fold[idx] = np.arange(len(idx)) % n_folds
    return fold


def crossfit(kind, Xt, y_np, tr_idx, eval_idx, slices, cfg, n_folds=5, seed=0, device="cpu",
             score_mask=None):
    """K-fold cross-fit one model family.

    kind: "encoder" (EncoderOnly) or "retrieval" (Retrieval; the pool is the
    fold's inner-train rows, so fold rows are scored exactly like held-out
    patients). Early stopping on the held-out fold. Returns dict with
      oof   [n_exits, len(tr_idx)]  out-of-fold logits for training rows,
      held  [n_exits, len(eval_idx)] fold-averaged logits for eval rows,
      folds, info (per-fold fit info), oof_auroc (last exit).
    `score_mask` (bool over all rows, optional): only these rows are used for
    early stopping and `oof_auroc` -- e.g. train on core + auxiliary patients
    but select on core patients, whose distribution matches the test set.
    """
    if score_mask is None:
        score_mask = np.ones(len(y_np), dtype=bool)
    y_t = torch.tensor(y_np, dtype=torch.float32, device=device)
    fold = stratified_folds(y_np[tr_idx], n_folds, seed)
    oof, held, infos = None, None, []
    for f in range(n_folds):
        torch.manual_seed(seed * 100 + f)
        inner, out = tr_idx[fold != f], tr_idx[fold == f]
        mk = dict(d=cfg.get("d", 128), p=cfg.get("p", 0.2), hid=cfg.get("hid", 128),
                  cont=cfg.get("cont"), plr=cfg.get("plr"))
        if kind == "encoder":
            model = EncoderOnly(slices, **mk).to(device)
            if cfg.get("init_enc") is not None:
                model.enc.load_state_dict(cfg["init_enc"])
            fwd = lambda b, train=True: model(Xt[b])
            bs = cfg.get("bs", 512)
        else:
            model = Retrieval(slices, m=cfg.get("m", 32), **mk).to(device)
            if cfg.get("init_enc") is not None:
                model.enc.load_state_dict(cfg["init_enc"])
            pool_pos = torch.full((len(y_np),), -1, dtype=torch.long, device=device)
            pool_pos[torch.as_tensor(inner, device=device)] = torch.arange(len(inner), device=device)
            x_pool, y_pool = Xt[inner], y_t[inner].long()
            fwd = (lambda b, train=True:
                   model(Xt[b], pool_pos[torch.as_tensor(b, device=device)] if train else None,
                         x_pool, y_pool))
            bs = cfg.get("bs", 256)
        out_s = out[score_mask[out]]
        vf = lambda: auroc(y_np[out_s], predict_rows(fwd, model, out_s, bs=1024)[-1])
        info = fit_minibatch(model, fwd, inner, y_t, vf, lr=cfg.get("lr", 1e-3),
                             wd=cfg.get("wd", 1e-4), bs=bs, patience=cfg.get("patience", 12),
                             max_ep=cfg.get("max_ep", 200), seed=seed * 100 + f)
        po = np.stack(predict_rows(fwd, model, out, bs=1024))
        ph = np.stack(predict_rows(fwd, model, eval_idx, bs=1024))
        if oof is None:
            oof = np.zeros((po.shape[0], len(tr_idx)))
            held = np.zeros((ph.shape[0], len(eval_idx)))
        oof[:, fold == f] = po
        held += ph / n_folds
        infos.append(info)
        del model
    return {"oof": oof, "held": held, "folds": fold, "info": infos,
            "oof_auroc": auroc(y_np[tr_idx][score_mask[tr_idx]], oof[-1][score_mask[tr_idx]])}


# ------------------------------------------- self-supervised pretraining

def pretrain_encoder(slices, Xt, rows, cfg, epochs=40, mask_p=0.3, bs=512, lr=1e-3, seed=0,
                     device="cpu"):
    """Masked-feature reconstruction (DuETT-style value+presence masking,
    tabular form): a random `mask_p` of every row's columns -- values AND
    missingness indicators -- is zeroed, and a linear decoder on h must
    reconstruct the original row (MSE on masked entries). Uses FEATURES of
    `rows` only, never labels. Returns the Encoder state_dict."""
    torch.manual_seed(seed)
    mk = dict(d=cfg.get("d", 128), p=cfg.get("p", 0.2), hid=cfg.get("hid", 128),
              cont=cfg.get("cont"), plr=cfg.get("plr"))
    enc = Encoder(slices, **mk).to(device)
    dec = nn.Linear(mk["d"], Xt.shape[1]).to(device)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(dec.parameters()), lr=lr, weight_decay=1e-4)
    rng = np.random.default_rng(seed)
    g = torch.Generator(device=device).manual_seed(seed)
    losses = []
    for ep in range(epochs):
        enc.train(); tot, nb = 0.0, 0
        perm = rng.permutation(rows)
        for s in range(0, len(perm), bs):
            x = Xt[perm[s:s + bs]]
            m = (torch.rand(x.shape, generator=g, device=device) < mask_p).float()
            rec = dec(enc(x * (1 - m)))
            loss = (((rec - x) ** 2) * m).sum() / m.sum().clamp_min(1)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        losses.append(tot / max(nb, 1))
    return {k: v.detach().clone() for k, v in enc.state_dict().items()}, losses


# ------------------------------------------------------ Correct & Smooth

def knn_train_graph(Z, tr_idx, q_idx, k, device="cpu", chunk=4096):
    """Cosine kNN: every query row -> its k nearest TRAINING rows. When a
    query is itself a training row, it is excluded from its own neighbours.
    Returns (nbr [len(q_idx), k] as positions into tr_idx, sim [.., k])."""
    Zt = F.normalize(torch.as_tensor(Z, device=device, dtype=torch.float32), dim=1)
    pool = Zt[torch.as_tensor(tr_idx, device=device)]
    pos = {int(t): i for i, t in enumerate(tr_idx)}
    nbrs, sims = [], []
    for s in range(0, len(q_idx), chunk):
        qi = q_idx[s:s + chunk]
        sim = Zt[torch.as_tensor(qi, device=device)] @ pool.T
        own = [(r, pos[int(q)]) for r, q in enumerate(qi) if int(q) in pos]
        if own:
            r, c = zip(*own)
            sim[torch.as_tensor(r, device=device), torch.as_tensor(c, device=device)] = -2.0
        v, i = sim.topk(k, dim=1)
        nbrs.append(i.cpu().numpy())
        sims.append(v.cpu().numpy())
    return np.concatenate(nbrs), np.concatenate(sims)


def cs_features(nbr_q, sim_q, nbr_tr, sim_tr, q_pos_in_tr, resid_tr, y_tr, tau=0.05):
    """Leak-free C&S features for query rows.

    nbr_q/sim_q:   the queries' k nearest training rows (self already excluded).
    nbr_tr/sim_tr: every training row's k nearest training rows (self excluded).
    q_pos_in_tr:   for each query, its own position in tr_idx, or -1 if held-out.
    resid_tr:      OOF residual y - p for every training row; y_tr: labels.

    Features: [r1, y1, r2, y2] = softmax(sim/tau)-weighted 1-hop and 2-hop
    means of neighbour residuals / labels. 2-hop paths q -> j -> q are removed
    (and the remaining weight renormalised), so a training query never reads
    its own residual or label through a cycle.
    """
    def w_of(sim):
        w = np.exp((sim - sim.max(1, keepdims=True)) / tau)
        return w / w.sum(1, keepdims=True)

    wq, wt = w_of(sim_q), w_of(sim_tr)
    r1 = (wq * resid_tr[nbr_q]).sum(1)
    y1 = (wq * y_tr[nbr_q]).sum(1)
    # 2-hop: weights W2[q, j, l] = wq[q, j] * wt[j, l] over l = nbr_tr[j]
    nb2 = nbr_tr[nbr_q]                                   # [nq, k, k]
    w2 = wq[:, :, None] * wt[nbr_q]                       # [nq, k, k]
    self_ret = (nb2 == q_pos_in_tr[:, None, None]) & (q_pos_in_tr[:, None, None] >= 0)
    w2 = np.where(self_ret, 0.0, w2)
    w2 = w2 / np.clip(w2.sum((1, 2), keepdims=True), 1e-12, None)
    r2 = (w2 * resid_tr[nb2]).sum((1, 2))
    y2 = (w2 * y_tr[nb2]).sum((1, 2))
    return np.stack([r1, y1, r2, y2], 1)


def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
%%writefile /kaggle/working/src/temporal_block.py
"""Time-binned trajectory features for labs and vitals (notebook 15).

`mimic_iv._aggregate_events` summarises each concept over the whole
observation window (first/last/min/max/mean/count). YAIB and DuETT
(RESEARCH.md prompt 1) both find the *trajectory* inside the window carries
signal those whole-window summaries lose. This adds, per concept:

    <prefix>_<concept>_b{0..n_bins-1}   mean inside each equal-width time bin
    <prefix>_<concept>_slope           least-squares slope, value per hour
    <prefix>_<concept>_hrs_since_last  window_end minus the last measurement, hours

Same window discipline as `mimic_iv` (events strictly inside
[intime, window_end)), same per-concept plausibility ranges
(`mimic_concepts` items: (concept, unit, lo, hi)), multi-itemid concepts
pooled at the concept level. Needs DuckDB.
"""

import numpy as np
import pandas as pd

from src import mimic_iv


def temporal_block(root, df, rel, id_col, time_col, value_col, items, prefix,
                   window_h=24.0, n_bins=4, verbose=True):
    """-> DataFrame [len(df), n_concepts * (n_bins + 2)], rows aligned with df."""
    concepts = sorted({v[0] for v in items.values()})
    cols = [f"{prefix}_{c}_{s}" for c in concepts
            for s in [f"b{b}" for b in range(n_bins)] + ["slope", "hrs_since_last"]]
    out = pd.DataFrame(np.nan, index=df.index, columns=cols, dtype=np.float32)
    path = mimic_iv.table_path(root, rel)
    if path is None:
        if verbose:
            print(f"  [{prefix}_t] {rel} absent -- block skipped")
        return out
    con = mimic_iv._duckdb()
    assert con is not None, "temporal_block needs duckdb"

    win = df[[id_col, "intime", "window_end"]].copy()
    win["row_id"] = np.arange(len(df))
    rng = pd.DataFrame([(int(i), v[0], float(v[2]), float(v[3])) for i, v in items.items()],
                       columns=["itemid", "concept", "lo", "hi"])
    con.register("win", win)
    con.register("rng", rng)
    bw = window_h / n_bins
    bins = ",\n".join(
        f"avg(CASE WHEN hr >= {b * bw} AND hr < {(b + 1) * bw} THEN v END) AS b{b}"
        for b in range(n_bins))
    sql = f"""
        WITH ev AS (
            SELECT w.row_id, r.concept, CAST(e.{value_col} AS DOUBLE) AS v,
                   epoch(CAST(e.{time_col} AS TIMESTAMP) - CAST(w.intime AS TIMESTAMP)) / 3600.0 AS hr
            FROM read_csv_auto('{path}', ignore_errors=true) e
            JOIN win w ON e.{id_col} = w.{id_col}
            JOIN rng r ON e.itemid = r.itemid
            WHERE e.{value_col} IS NOT NULL
              AND e.{time_col} >= w.intime AND e.{time_col} < w.window_end
              AND e.{value_col} BETWEEN r.lo AND r.hi
        )
        SELECT row_id, concept, {bins},
               regr_slope(v, hr) AS slope,
               {window_h} - max(hr) AS hrs_since_last
        FROM ev GROUP BY row_id, concept
    """
    if verbose:
        print(f"  [{prefix}_t] duckdb scan of {rel} ...", flush=True)
    agg = con.execute(sql).fetch_df()
    con.close()
    for concept, blk in agg.groupby("concept"):
        rows = blk["row_id"].to_numpy()
        for s in [f"b{b}" for b in range(n_bins)] + ["slope", "hrs_since_last"]:
            out.iloc[rows, out.columns.get_loc(f"{prefix}_{concept}_{s}")] = \
                blk[s].to_numpy(dtype=np.float64).astype(np.float32)
    if verbose:
        print(f"  [{prefix}_t] {len(cols)} cols, {len(agg):,} (stay, concept) groups", flush=True)
    return out.astype(np.float32)

In [ ]:
%%writefile /kaggle/working/tests/test_graph_v2.py
"""Leakage invariants for src/graph_v2.py (run inside the Kaggle notebooks)."""

import numpy as np
import torch

from src import graph_v2 as g


def _toy(n=60, d=5, seed=0):
    rng = np.random.default_rng(seed)
    Z = rng.normal(size=(n, d)).astype(np.float32)
    y = (rng.random(n) < 0.3).astype(float)
    return Z, y


def test_knn_train_graph_excludes_self():
    Z, _ = _toy()
    tr = np.arange(40)
    nbr, _ = g.knn_train_graph(Z, tr, tr, k=5)
    assert not (nbr == np.arange(40)[:, None]).any()


def test_cs_features_never_read_own_label_or_residual():
    Z, y = _toy()
    tr = np.arange(40)
    resid = y[tr] - 0.3
    nbr_tr, sim_tr = g.knn_train_graph(Z, tr, tr, k=6)
    q_pos = np.arange(40)
    base = g.cs_features(nbr_tr, sim_tr, nbr_tr, sim_tr, q_pos, resid, y[tr])
    for i in (0, 7, 23):
        r2, y2 = resid.copy(), y[tr].copy()
        r2[i] += 5.0
        y2[i] = 1 - y2[i]
        pert = g.cs_features(nbr_tr, sim_tr, nbr_tr, sim_tr, q_pos, r2, y2)
        assert np.allclose(pert[i], base[i]), f"row {i} reads its own label/residual"


def test_heldout_queries_read_only_training_rows():
    Z, y = _toy()
    tr, te = np.arange(40), np.arange(40, 60)
    nbr, _ = g.knn_train_graph(Z, tr, te, k=5)
    assert nbr.max() < len(tr) and nbr.min() >= 0


def test_stratified_folds_cover_every_row_once():
    y = np.array([0] * 30 + [1] * 10)
    f = g.stratified_folds(y, 5, 0)
    assert set(f) == set(range(5))
    for k in range(5):
        assert (y[f == k] == 1).sum() == 2


def test_retrieval_excludes_self_from_context():
    torch.manual_seed(0)
    sl = {"a": (0, 3), "b": (3, 5)}
    x = torch.randn(20, 5)
    yv = torch.zeros(20, dtype=torch.long)
    yv[0] = 1
    m = g.Retrieval(sl, m=19, d=8, hid=8)
    m.eval()
    # query 0 is pool member 0; with m = n_pool - 1 its context is everyone else,
    # so flipping its own label must not change its retrieval exit.
    a = m(x[:1], torch.tensor([0]), x, yv)[1]
    yv2 = yv.clone(); yv2[0] = 0
    b = m(x[:1], torch.tensor([0]), x, yv2)[1]
    assert torch.allclose(a, b)


def test_plr_encoder_shapes_and_grad():
    rng = np.random.default_rng(0)
    X = np.where(rng.random((50, 6)) < 0.2, np.nan, rng.normal(size=(50, 6)))
    X[:, 5] = (rng.random(50) < 0.5).astype(float)  # binary column
    Xp, sl, cont = g.prep_views(X, {"a": np.arange(4), "b": np.arange(4, 6)},
                                np.arange(50) < 40, "iqr", True, return_cont=True)
    assert cont["b"].sum() == 1  # column 4 continuous, column 5 binary, rest indicators
    m = g.EncoderOnly(sl, d=8, hid=8, cont=cont, plr={"n_freq": 4, "sigma": 0.1, "e": 3})
    out = m(torch.tensor(Xp))[0]
    assert out.shape == (50,)
    out.sum().backward()
    assert m.enc.plr["a"].c.grad is not None


def test_pretrain_then_load_into_crossfit_encoder():
    rng = np.random.default_rng(1)
    X = np.where(rng.random((80, 6)) < 0.2, np.nan, rng.normal(size=(80, 6)))
    y = (rng.random(80) < 0.3).astype(np.int64)
    tr = np.arange(60)
    Xp, sl = g.prep_views(X, {"a": np.arange(4), "b": np.arange(4, 6)}, np.arange(80) < 60, "iqr", True)
    Xt = torch.tensor(Xp)
    cfg = dict(d=8, hid=8, p=0.1)
    st, losses = g.pretrain_encoder(sl, Xt, tr, cfg, epochs=3, bs=16)
    assert len(losses) == 3 and np.isfinite(losses).all()
    cf = g.crossfit("encoder", Xt, y, tr, np.arange(60, 80), sl, dict(cfg, init_enc=st, max_ep=2),
                    n_folds=3, seed=0)
    assert cf["oof"].shape == (1, 60)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", f"{WORK}/tests/test_graph_v2.py"],
                   cwd=WORK, capture_output=True, text=True, env={**os.environ, "PYTHONPATH": WORK})
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "leakage tests failed -- refusing to run"
print("[v2] leakage tests passed", flush=True)

## Cohorts: core (cached) + full (block caches reused from notebook 12's output)

In [ ]:
hits = glob.glob("/kaggle/input/**/processed/cohort_97de93de81.csv.gz", recursive=True)
assert hits, "attach kernel satgunsinghsodhi/graph-v2-label-aware-experiments (its processed/ output)"
src_dir = os.path.dirname(hits[0])
for f in os.listdir(src_dir):
    if f.startswith(("cohort_97de93de81", "block_")):
        shutil.copy(os.path.join(src_dir, f), f"{WORK}/processed/{f}")
print("[v2] cached files:", sorted(os.listdir(f"{WORK}/processed")), flush=True)


def _find(start, max_depth=6):
    for dp, dn, _ in os.walk(start):
        if "hosp" in dn:
            b = os.path.basename(dp)
            return os.path.dirname(dp) if b in ("mimic-iv-3.1", "mimic-iv") else dp
        if dp[len(start):].count(os.sep) >= max_depth:
            dn[:] = []
    return None


RAW = _find("/kaggle/input")
if RAW is None:
    import kagglehub
    _dl = kagglehub.dataset_download("akshaybe/updated-mimic-iv")
    RAW = _find(_dl, 3) or _dl
print("[v2] MIMIC root:", RAW, flush=True)

from src import mimic_iv


def cfg_for(required):
    return {"seed": 42, "data": {
        "source": "mimic", "task": "mortality", "raw_dir": RAW, "cache_dir": f"{WORK}/processed",
        "obs_window_hours": 24, "min_los_hours": 28, "max_patients": None, "cohort_seed": 42,
        "force_rebuild": False, "blocks": ["demographics", "comorbidity", "labs", "vitals", "treatment"],
        "require_nonempty_blocks": required}}


CFG_CORE, CFG_FULL = cfg_for(["vitals"]), cfg_for([])
assert mimic_iv.cohort_signature(CFG_CORE) == "97de93de81"
t0 = time.time()
Xc, Yc, Nc, Mc = mimic_iv.build_cohort(CFG_CORE)
Xf, Yf, Nf, Mf = mimic_iv.build_cohort(CFG_FULL)
print(f"[v2] core n={len(Yc)}  full n={len(Yf)}  ({time.time()-t0:.0f}s)", flush=True)
assert list(Nc) == list(Nf), "feature columns differ between core and full cohorts"

# locate core rows inside the full cohort (order-preserving filter, asserted)
pos = {s: i for i, s in enumerate(Mf["stay_id"])}
CORE_IN_FULL = np.array([pos[s] for s in Mc["stay_id"]])
assert np.all(np.diff(CORE_IN_FULL) > 0), "core rows are not an order-preserving subset of full"
assert np.array_equal(Yf[CORE_IN_FULL], Yc)
AUX = np.setdiff1d(np.arange(len(Yf)), CORE_IN_FULL)
X_RAW, Y = Xf, Yf.astype(np.int64)
VIEWS = {k: np.asarray(v) for k, v in Mf["views"].items()}
IS_CORE = np.zeros(len(Y), dtype=bool); IS_CORE[CORE_IN_FULL] = True
print(f"[v2] aux patients={len(AUX)} prevalence core={Yc.mean():.4f} aux={Y[AUX].mean():.4f}", flush=True)

## Temporal block (labs ~2 min + chartevents ~4 min DuckDB scans), cached to output

In [ ]:
from src import temporal_block as tb
from src import mimic_concepts as mc
TPATH = f"{WORK}/processed/temporal_full_v1.parquet"
t0 = time.time()
base_df, _ = mimic_iv._base_cohort(mimic_iv.find_root(RAW), CFG_FULL)
base_df = base_df.reset_index(drop=True)
assert np.array_equal(base_df["stay_id"].to_numpy(), Mf["stay_id"]), "base cohort order != full cohort"
hit_t = glob.glob("/kaggle/input/**/temporal_full_v1.parquet", recursive=True)
if hit_t:
    TB = pd.read_parquet(hit_t[0])
else:
    lab_t = tb.temporal_block(mimic_iv.find_root(RAW), base_df, "hosp/labevents.csv",
                              "hadm_id", "charttime", "valuenum", mc.LAB_ITEMS, "labt")
    vit_t = tb.temporal_block(mimic_iv.find_root(RAW), base_df, "icu/chartevents.csv",
                              "stay_id", "charttime", "valuenum", mc.CHART_ITEMS, "vitt")
    TB = pd.concat([lab_t, vit_t], axis=1)
    TB.to_parquet(TPATH)
assert len(TB) == len(Y)
X_TEMP = TB.to_numpy(np.float32)
n_base = X_RAW.shape[1]
lab_cols = [i for i, c in enumerate(TB.columns) if c.startswith("labt_")]
vit_cols = [i for i, c in enumerate(TB.columns) if c.startswith("vitt_")]
X_BT = np.concatenate([X_RAW, X_TEMP], 1)
VIEWS_BT = {**VIEWS, "labs_t": n_base + np.array(lab_cols), "vitals_t": n_base + np.array(vit_cols)}
FEATS = {"base": (X_RAW, VIEWS), "base+temporal": (X_BT, VIEWS_BT)}
print(f"[v2] temporal block {X_TEMP.shape} nan-frac={np.isnan(X_TEMP).mean():.3f} "
      f"(core rows {np.isnan(X_TEMP[IS_CORE]).mean():.3f}) ({time.time()-t0:.0f}s)", flush=True)

## Splits on the CORE cohort (verbatim notebook 10-13 logic), mapped into full indices

In [ ]:
def make_splits(y, seed, fracs=(0.6, 0.2, 0.2)):
    rng = np.random.default_rng(seed)
    n = len(y)
    train, val, test = (np.zeros(n, dtype=bool) for _ in range(3))
    for cls in np.unique(y):
        idx = np.flatnonzero(y == cls)
        rng.shuffle(idx)
        n_tr = int(round(fracs[0] * len(idx))); n_va = int(round(fracs[1] * len(idx)))
        train[idx[:n_tr]] = True; val[idx[n_tr:n_tr + n_va]] = True; test[idx[n_tr + n_va:]] = True
    return train, val, test


def split_validation_3way(val_mask, y, seed, fracs=(0.4, 0.3, 0.3)):
    rng = np.random.default_rng(seed + 992)
    parts = [np.zeros_like(val_mask) for _ in range(3)]
    idx = np.flatnonzero(val_mask)
    for cls in (0, 1):
        c = rng.permutation(idx[y[idx] == cls])
        cut1 = int(round(fracs[0] * len(c))); cut2 = cut1 + int(round(fracs[1] * len(c)))
        parts[0][c[:cut1]] = True; parts[1][c[cut1:cut2]] = True; parts[2][c[cut2:]] = True
    return parts

## Run: scratch vs pretrained encoder, bagged winner, G1 from the pretrained encoder

In [ ]:
from xgboost import XGBClassifier
from src import graph_v2 as g

ALL = {}
ENC_CFGS = {
    "B_lr3e-4_wd1e-3_p0.3": dict(lr=3e-4, wd=1e-3, p=0.3, d=128, hid=128),
    "C_lr3e-4_wd1e-2_p0.4_d256": dict(lr=3e-4, wd=1e-2, p=0.4, d=256, hid=256),
}


def rep(seed, name, row):
    print(f"[v2][seed {seed}] {name:40s} " + " ".join(
        f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items()), flush=True)
    ALL.setdefault(str(seed), {})[name] = row
    with open(f"{OUT}/results_v2_round6.json", "w") as f:
        json.dump(ALL, f, indent=1)


PLR_CFGS = {"none": None,
            "plr_s0.05": {"n_freq": 16, "sigma": 0.05, "e": 8},
            "plr_s0.3": {"n_freq": 16, "sigma": 0.3, "e": 8}}
BASE_CFG = ENC_CFGS["C_lr3e-4_wd1e-2_p0.4_d256"]
X_RAW, VIEWS = FEATS["base+temporal"]

for SEED in SEEDS:
    t_seed = time.time()
    np.random.seed(SEED); torch.manual_seed(SEED)
    tr_c, va_c, te_c = make_splits(Yc.astype(np.int64), SEED)
    mv_c, _, _ = split_validation_3way(va_c, Yc.astype(np.int64), SEED)
    tr_core = CORE_IN_FULL[tr_c]
    mv_idx, te_idx = CORE_IN_FULL[mv_c], CORE_IN_FULL[te_c]
    ev_idx = np.concatenate([mv_idx, te_idx])
    mv_pos, te_pos = np.arange(len(mv_idx)), np.arange(len(mv_idx), len(ev_idx))
    S = lambda z: {"model_val_auroc": g.auroc(Y[mv_idx], z[mv_pos]),
                   "test_auroc": g.auroc(Y[te_idx], z[te_pos]),
                   "test_auprc": float(average_precision_score(Y[te_idx], z[te_pos]))}
    tr_idx = np.sort(np.concatenate([tr_core, AUX]))

    clf = XGBClassifier(n_estimators=4000, max_depth=5, learning_rate=0.02, subsample=0.8,
                        colsample_bytree=0.5, min_child_weight=2, reg_lambda=2.0, eval_metric="auc",
                        early_stopping_rounds=300, random_state=SEED, n_jobs=4, tree_method="hist",
                        device="cuda" if DEV.type == "cuda" else "cpu")
    clf.fit(X_RAW[tr_idx], Y[tr_idx], eval_set=[(X_RAW[mv_idx], Y[mv_idx])], verbose=False)
    rep(SEED, "REF_xgb_tuned[base+temporal]", S(clf.predict_proba(X_RAW[ev_idx])[:, 1]))

    tr_mask = np.zeros(len(Y), dtype=bool); tr_mask[tr_idx] = True
    Xp, SL = g.prep_views(X_RAW, VIEWS, tr_mask, "iqr", True)
    Xt = torch.tensor(Xp, device=DEV)

    t0 = time.time()
    PRE, pre_losses = g.pretrain_encoder(SL, Xt, tr_idx, BASE_CFG, epochs=40, mask_p=0.3,
                                         seed=SEED, device=DEV)
    print(f"[v2][seed {SEED}] pretrain loss {pre_losses[0]:.4f} -> {pre_losses[-1]:.4f} "
          f"({time.time()-t0:.0f}s)", flush=True)
    ARMS = {"scratch": dict(BASE_CFG), "pretrained": dict(BASE_CFG, init_enc=PRE)}
    best = None
    for aname, cfg in ARMS.items():
        cf = g.crossfit("encoder", Xt, Y, tr_idx, ev_idx, SL, cfg, n_folds=5, seed=SEED * 10,
                        device=DEV, score_mask=IS_CORE)
        rep(SEED, f"ENC[{aname}]", {"oof_auroc_core": cf["oof_auroc"], **S(cf["held"][-1]),
                                    "best_epochs": [i["best_epoch"] for i in cf["info"]]})
        if best is None or cf["oof_auroc"] > best[0]:
            best = (cf["oof_auroc"], aname, cf)
    helds = [best[2]["held"][-1]]
    for b in (1, 2):
        cfb = g.crossfit("encoder", Xt, Y, tr_idx, ev_idx, SL, ARMS[best[1]], n_folds=5,
                         seed=SEED * 10 + b, device=DEV, score_mask=IS_CORE)
        helds.append(cfb["held"][-1])
    enc_held = np.mean(helds, 0)
    rep(SEED, f"ENC_bag3[{best[1]}]", S(enc_held))

    rcfg = dict(ARMS["pretrained"], m=32, lr=3e-4, patience=8)
    rf = g.crossfit("retrieval", Xt, Y, tr_idx, ev_idx, SL, rcfg, n_folds=5, seed=SEED * 10 + 5,
                    device=DEV, score_mask=IS_CORE)
    rep(SEED, "G1[pretrained][m=32]", {"oof_auroc_core": rf["oof_auroc"], **S(rf["held"][-1]),
                                       "exit0_test_auroc": g.auroc(Y[te_idx], rf["held"][0][te_pos])})
    rep(SEED, f"ENC_bag3[{best[1]}]+G1[pretrained]", S((enc_held + rf["held"][-1]) / 2))
    del Xt
    torch.cuda.empty_cache()
    print(f"[v2][seed {SEED}] done in {time.time()-t_seed:.0f}s", flush=True)

names = sorted({n for s in ALL.values() for n in s})
print("\n[v2] SUMMARY test AUROC mean ± sd over seeds", SEEDS)
for n in names:
    v = [ALL[str(s)][n]["test_auroc"] for s in SEEDS if n in ALL[str(s)]]
    print(f"[v2] {n:46s} {np.mean(v):.4f} ± {np.std(v):.4f}  per-seed={[round(x, 4) for x in v]}")
print("[v2] DONE", flush=True)